In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
from model.expression_embedding import expression_GVAE
from common.data_processing import data_preprocessing_ST1K

from mclSTExp.train import main

In [ ]:
# input_path='./data/HEST/SKIN-PS-NL/'
# input_path='./data/HEST/SKIN-LP-NL/'
# input_path='./data/HEST/SKIN-AD-NL/'
# adata_list, HE_image_paths,nuclei_mask_paths=data_preprocessing_ST1K(input_path)
# names = [d[:-5] for d in os.listdir(input_path) if d.endswith('.h5ad')]

input_path='./data/HEST/COLON-CANCER_Xenium/'
names = [d[:-5] for d in os.listdir(input_path) if d.endswith('.h5ad')]
adata_list, HE_image_paths, nuclei_mask_paths=data_preprocessing_ST1K(input_path)

In [ ]:

for i in range(len(adata_list)):
    sc.pp.filter_genes(adata_list[i], min_cells=1)
    sc.pp.filter_cells(adata_list[i], min_genes=1)
# 计算所有anndata共享的var_names
shared_var_names = set(adata_list[0].var_names)
for adata in adata_list[1:]:
    shared_var_names.intersection_update(adata.var_names)
print(len(shared_var_names))

for i in range(len(adata_list)):
    adata_list[i] = adata_list[i][:, list(shared_var_names)]
    
print(adata_list[0])

In [ ]:
Input_Method='hvg'
new_adata_list=expression_GVAE(adatas=adata_list,method='GATE',feature_method=Input_Method,n_top_genes=100)
print(new_adata_list[0])

In [ ]:
out_embedding=new_adata_list[0].X.shape[1]
for i in range(len(new_adata_list)):
    if 'block_r' in adata_list[i].uns and adata_list[i].uns['block_r'] > 32:
        new_adata_list[i].uns['block_r']=adata_list[i].uns['block_r']
        print(new_adata_list[i].uns['block_r'])
    else:
        new_adata_list[i].uns['block_r']=112
print(out_embedding)

In [ ]:
from mclSTExp.model import mclSTExp_Attention
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F
from tqdm import tqdm
from mclSTExp.dataset import HERDataset
from scipy.sparse import issparse

def find_matches(spot_embeddings, query_embeddings, top_k=1):
    #find the closest matches 
    spot_embeddings = torch.tensor(spot_embeddings)
    query_embeddings = torch.tensor(query_embeddings)
    query_embeddings = F.normalize(query_embeddings, p=2, dim=-1)
    spot_embeddings = F.normalize(spot_embeddings, p=2, dim=-1)
    dot_similarity = query_embeddings @ spot_embeddings.T   #2277x2265
    _, indices = torch.topk(dot_similarity.squeeze(0), k=top_k)
    
    return indices.cpu().numpy()

train_index=[i for i in range(len(new_adata_list))]

for i in train_index: 
    test_index=[i]
    new_train_index = [item for item in train_index if item not in test_index]
    savepath=os.path.join(input_path,names[test_index[0]],'results/')
    if os.path.exists(input_path+names[test_index[0]]+'/results/mclSTExp_pre.h5ad'):
        print('skip')
        continue

    main(new_adata_list,HE_image_paths,names,new_train_index,test_index,out_embedding,save_path=savepath)
    
    model_path = savepath + "/mclSTExp_best.pt"
    device = torch.device("cuda:2")
    model = mclSTExp_Attention(encoder_name='densenet121',
                                spot_dim=out_embedding,
                                temperature=1,
                                image_dim=1024,
                                projection_dim=256,
                                heads_num=8,
                                heads_dim=64,
                                head_layers=2,
                                dropout=0).to(device)  
    state_dict = torch.load(model_path)
    new_state_dict = {}
    for key in state_dict.keys():
        new_key = key.replace('module.', '')  # remove the prefix 'module.'
        new_key = new_key.replace('well', 'spot') # for compatibility with prior naming
        new_state_dict[new_key] = state_dict[key]

    model.load_state_dict(new_state_dict)
    _=model.eval()
    train_HE_image_paths=[HE_image_paths[i] for i in new_train_index]
    train_adata_list=[new_adata_list[i] for i in new_train_index]
    train_names=[names[i] for i in new_train_index]
    train_dataset = HERDataset(train_HE_image_paths, train_adata_list, train=True, names=train_names)
    train_dataLoader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=False)
    test_names=[names[i] for i in test_index]
    test_HE_image_paths=[HE_image_paths[i] for i in test_index]
    test_adata_list=[new_adata_list[i] for i in test_index]    
    test_dataset = HERDataset(test_HE_image_paths, test_adata_list, train=False, names=test_names)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0, drop_last=False)
    test_image_embeddings = []
    train_spot_embeddings = []   
    with torch.no_grad():
        for batch in tqdm(train_dataLoader):
            spot_feature = batch["expression"].to(device)
            x = batch["position"][:, 0].long().to(device)
            y = batch["position"][:, 1].long().to(device)
            centers_x = model.x_embed(x)
            centers_y = model.y_embed(y)
            spot_feature = spot_feature + centers_x + centers_y
            spot_features = spot_feature.unsqueeze(dim=0)
            spot_embedding = model.spot_encoder(spot_features)
            spot_embedding = model.spot_projection(spot_embedding).squeeze(dim=0)
            train_spot_embeddings.append(spot_embedding)
        for batch in tqdm(test_loader):
            image_features = model.image_encoder(batch["image"].to(device))
            image_embeddings = model.image_projection(image_features)
            test_image_embeddings.append(image_embeddings)  
            
    img_embeddings_alltest = torch.cat(test_image_embeddings, dim=0)
    spot_embeddings_alltrain = torch.cat(train_spot_embeddings, dim=0)

    img_embeddings_alltest = img_embeddings_alltest.cpu().numpy()
    spot_embeddings_alltrain = spot_embeddings_alltrain.cpu().numpy()
    print(img_embeddings_alltest.shape)
    print(spot_embeddings_alltrain.shape)

    spot_expression=[]
    for k in new_train_index:
        if issparse(new_adata_list[k].X):
            adata_array=np.array(new_adata_list[k].X.todense())
        else:
            adata_array=np.array(new_adata_list[k].X)
        spot_expression.append(adata_array)

    expression_key=np.concatenate(spot_expression, axis = 0)
    print(expression_key.shape)

    indices = find_matches(spot_embeddings_alltrain, img_embeddings_alltest, top_k=50)
    matched_spot_expression_pred = np.zeros((indices.shape[0], expression_key.shape[1]))
    for k in range(indices.shape[0]):
        matched_spot_expression_pred[k,:] = np.average(expression_key[indices[k,:],:], axis=0)
        
    gt_adata=new_adata_list[test_index[0]].copy()
    predict_adata=new_adata_list[test_index[0]].copy()
    predict_adata.X=matched_spot_expression_pred.copy()
    predict_adata.write_h5ad(input_path+names[test_index[0]]+'/results/mclSTExp_pre.h5ad')
    gt_adata.write_h5ad(input_path+names[test_index[0]]+'/results/mclSTExp_gt.h5ad')   